# Storm Stations + Best Track (single case)

**One unified map of a storm's validation stations and its NOAA best track.**

This notebook takes an **explicit hydrograph package path** (the stations) and an **explicit
ATCF track `.dat` path**, then draws both on a single map. Stations are agency-coded
(triangles = observed data, circles = forecast-only) with Agency and Data legends.

Runs locally and on Google Colab. On Colab (or any machine without the files) you can set the
optional download links and the notebook fetches the data first.

**Pipeline:**

```
hydrograph package (stations) + ATCF .dat (track)  ->  one map  ->  outputs/station_track_plots/*.png
```


## 0. Environment

Colab does not ship `cartopy`; install it there on first use.

In [10]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    try:
        import cartopy  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cartopy"], check=False)

print("IN_COLAB:", IN_COLAB)

IN_COLAB: False


## 1. Setup --- imports, constants, helpers

`AGENCY_COLORS` is the publication agency palette. The helpers read station metadata from the
hydrograph JSON files and the track from the ATCF `.dat`.

In [11]:
%matplotlib inline

import csv
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patheffects as pe
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import warnings
import cartopy.io
warnings.filterwarnings('ignore', category=cartopy.io.DownloadWarning)

# Publication agency palette (matches the station-map script).
AGENCY_COLORS = {
    "USGS":        "#2E7D32",
    "CPRA_USGS":   "#2E7D32",
    "NOAA_NOS":    "#F5A623",
    "USACE":       "#D62728",
    "CPRA_USACE":  "#D62728",
    "IOC-UNESCO":  "#1565C0",
    "TCOON":       "#009688",
    "NOAA_RFS":    "#8E24AA",
    "PRSN":        "#E91E63",
    "Unknown":     "#D62728",
}
AGENCY_ORDER = ["USGS", "CPRA_USGS", "NOAA_NOS", "USACE", "CPRA_USACE",
                "TCOON", "IOC-UNESCO", "NOAA_RFS", "PRSN", "Unknown"]

ATCF_DATETIME_FORMAT = "%Y%m%d%H"


def load_stations(package_dir: Path) -> pd.DataFrame:
    """Read station metadata from a CERA hydrograph package."""
    hydro_dir = Path(package_dir) / "hydrographs"
    records = []
    for json_file in sorted(hydro_dir.glob("*.json")):
        with open(json_file) as f:
            data = json.load(f)
        records.append({
            "station_id": str(data.get("station_id", "")),
            "station_name": str(data.get("station_name", "")),
            "agency": str(data.get("agency", "Unknown")),
            "lat": data.get("lat"),
            "lon": data.get("lon"),
            "vertical_datum": str(data.get("vertical_datum", "")),
            "has_observed": bool(data.get("observed_data")),
        })
    if not records:
        raise ValueError(f"No station JSON files under {hydro_dir}")
    return pd.DataFrame(records)


def load_atcf_track(dat_path: Path) -> list:
    """Return the track as a list of (lat, lon) points from an ATCF DAT file."""
    rows = []
    with open(dat_path, newline="", encoding="utf-8") as f:
        for fields in csv.reader(f):
            fields = [c.strip() for c in fields]
            if len(fields) < 9:
                continue
            try:
                m = re.fullmatch(r"(\d+)([NSEW])", fields[6].upper())
                lat = int(m.group(1)) / 10.0 * (-1 if m.group(2) in ("S", "W") else 1)
                m = re.fullmatch(r"(\d+)([NSEW])", fields[7].upper())
                lon = int(m.group(1)) / 10.0 * (-1 if m.group(2) in ("S", "W") else 1)
                rows.append((lat, lon))
            except (AttributeError, ValueError):
                continue
    return rows


def padded_extent(stations: pd.DataFrame, track: list, min_pad: float = 0.75) -> list:
    """Extent covering both the stations and the track, with padding."""
    lons = list(stations["lon"]) + [p[1] for p in track]
    lats = list(stations["lat"]) + [p[0] for p in track]
    lon_min, lon_max, lat_min, lat_max = min(lons), max(lons), min(lats), max(lats)
    lon_pad = max((lon_max - lon_min) * 0.075, min_pad)
    lat_pad = max((lat_max - lat_min) * 0.075, min_pad)
    return [lon_min - lon_pad, lon_max + lon_pad, lat_min - lat_pad, lat_max + lat_pad]


def add_base_features(ax) -> None:
    ax.add_feature(cfeature.OCEAN.with_scale("10m"), facecolor="#D8DBDE", zorder=0)
    ax.add_feature(cfeature.LAND.with_scale("10m"), facecolor="#FFFFFF", zorder=0)
    ax.add_feature(cfeature.LAKES.with_scale("10m"), facecolor="#D8DBDE",
                   edgecolor="#A8A8A8", linewidth=0.25, zorder=1)
    ax.add_feature(cfeature.RIVERS.with_scale("10m"), edgecolor="#A8A8A8",
                   linewidth=0.22, alpha=0.75, zorder=1)
    ax.add_feature(cfeature.COASTLINE.with_scale("10m"), edgecolor="#A8A8A8",
                   linewidth=0.45, alpha=0.65, zorder=2)
    ax.add_feature(cfeature.BORDERS.with_scale("10m"), edgecolor="#CFCFCF",
                   linewidth=0.30, alpha=0.55, zorder=2)
    ax.add_feature(cfeature.STATES.with_scale("10m"), edgecolor="#CFCFCF",
                   linewidth=0.30, alpha=0.55, zorder=2)


def clean_path_part(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value).strip()).strip("_")

## 2. Paths

`OUTPUT_DIR` is chosen automatically: under `/content` on Colab, next to the notebook locally
(override with `HYDRO_WORKDIR`).

In [12]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

# Project root (holds code/ and data/). Detected via the always-present code/
# directory, so it is correct even before any data has been downloaded.
_cwd = Path.cwd().resolve()
if "HYDRO_WORKDIR" in os.environ:
    WORKDIR = Path(os.environ["HYDRO_WORKDIR"]).resolve()
elif IN_COLAB:
    WORKDIR = Path("/content/per_storm")
elif (_cwd.parent / "code").is_dir():
    WORKDIR = _cwd.parent          # run from .../code/
elif (_cwd / "code").is_dir():
    WORKDIR = _cwd                 # run from the project root
else:
    WORKDIR = _cwd

DATA_ROOT  = WORKDIR / "data"
OUTPUT_DIR = WORKDIR / "outputs" / "station_track_plots"

print("IN_COLAB  :", IN_COLAB)
print("WORKDIR   :", WORKDIR)
print("DATA_ROOT :", DATA_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)


IN_COLAB  : False
WORKDIR   : D:\Projects\Hindcast_archive\ipynb\plot_stations_track
DATA_ROOT : D:\Projects\Hindcast_archive\ipynb\plot_stations_track\data
OUTPUT_DIR: D:\Projects\Hindcast_archive\ipynb\plot_stations_track\outputs\station_track_plots


## 3. Input --- explicit paths

Edit the cell below. `PACKAGE_DIR` is the CERA hydrograph package (holds `hydrographs/*.json`);
`DAT_PATH` is the ATCF track. The optional URLs are only used when a path is missing
(e.g. on Colab).

In [13]:
# ----------------------------------------------------------------------
# INPUT (explicit paths for the MILTON 2024 case)
# ----------------------------------------------------------------------
PACKAGE_DIR = (DATA_ROOT / "hydrographs" / "2024" / "MILTON" / "EGOMv20b"
               / ("CERA_Hydrographs_TimeRange-2024100500-2024101018_"
                  "Storm-MILTON_Hindcast_BestTrack_ADCIRCMesh-EGOMv20b"))
DAT_PATH    = DATA_ROOT / "tracks" / "2024_MILTON" / "bal142024.dat"
TITLE       = "Milton (2024)"

DPI           = 300
OUTPUT_FORMAT = "png"

print("PACKAGE_DIR:", PACKAGE_DIR)
print("DAT_PATH   :", DAT_PATH)


PACKAGE_DIR: D:\Projects\Hindcast_archive\ipynb\plot_stations_track\data\hydrographs\2024\MILTON\EGOMv20b\CERA_Hydrographs_TimeRange-2024100500-2024101018_Storm-MILTON_Hindcast_BestTrack_ADCIRCMesh-EGOMv20b
DAT_PATH   : D:\Projects\Hindcast_archive\ipynb\plot_stations_track\data\tracks\2024_MILTON\bal142024.dat


## 4. Data acquisition

Uses the local package and track when present; otherwise downloads them (hydrographs archive
extracted into `DATA_DIR`, track written to `DAT_PATH`), with optional SHA-256 checks.

In [14]:
# ----------------------------------------------------------------------
# Data acquisition: fetch the single self-contained archive
# (hydrographs + tracks) if it is not already on disk.
#
#   DATA_ROOT/hydrographs/{YEAR}/{STORM}/{MESH}/CERA_Hydrographs_.../hydrographs/*.json
#   DATA_ROOT/tracks/{YEAR}_{STORM}/bal*.dat
#
# Runs on Google Colab and locally. If the data is already present it is skipped.
# ----------------------------------------------------------------------
import hashlib
import ssl
import urllib.request
import zipfile

# Single public download link for per_storm_data.zip (fill in for Colab).
DATA_URL    = "https://cloud.cera.lsu.edu/public.php/dav/files/cq8Tb34b63xjycY/?accept=zip"
# SHA-256 of that archive, verified after download to guard integrity.
DATA_SHA256 = "a090b33f56bab601674e3df092e8cea65604ce9656518b0ae7731e9ed8393657"


def _download(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    # The CERA host currently serves an expired TLS certificate, so verification
    # is disabled here; the SHA-256 check below is what protects integrity.
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    with urllib.request.urlopen(url, context=ctx) as resp, open(dest, "wb") as fh:
        fh.write(resp.read())


have_data = (DATA_ROOT / "hydrographs").is_dir() and (DATA_ROOT / "tracks").is_dir()
if have_data:
    print("Data already present:", DATA_ROOT)
elif DATA_URL:
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    zip_path = DATA_ROOT / "per_storm_data.zip"
    print("Downloading", DATA_URL, "->", zip_path)
    _download(DATA_URL, zip_path)
    if DATA_SHA256:
        digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
        assert digest == DATA_SHA256, f"checksum mismatch:\n got {digest}\n exp {DATA_SHA256}"
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_ROOT)   # archive top level: hydrographs/ + tracks/
    print("Extracted to", DATA_ROOT)
else:
    raise FileNotFoundError(
        f"Data missing under {DATA_ROOT} and DATA_URL is empty. "
        f"Set DATA_URL to the per_storm_data.zip link."
    )

# Sanity check.
print("Packages found:", len(list((DATA_ROOT / "hydrographs").glob("*/*/*/CERA_Hydrographs_*"))))
print("Track DATs found:", len(list((DATA_ROOT / "tracks").glob("*_*/bal*.dat"))))


Data already present: D:\Projects\Hindcast_archive\ipynb\plot_stations_track\data
Packages found: 13
Track DATs found: 13


## 5. Load stations and track

In [15]:
stations = load_stations(PACKAGE_DIR)
track = load_atcf_track(DAT_PATH)

print(f"Stations: {len(stations)}  (observed: {int(stations['has_observed'].sum())}, "
      f"forecast-only: {int((~stations['has_observed']).sum())})")
print("Agencies:", ", ".join(sorted(stations['agency'].unique())))
print("Track points:", len(track))

Stations: 144  (observed: 142, forecast-only: 2)
Agencies: NOAA_NOS, USGS
Track points: 58


## 6. Render the unified map

Black line = NOAA best track; markers = stations (agencies colored, triangles observed,
circles forecast-only). The map covers both the stations and the track.

In [ ]:
#def plot_stations_track(package_dir: Path, dat_path: Path, output_dir: Path,
#                        title: str = "", dpi: int = 300, output_format: str = "png"):
#def plot_stations_track(title: str = "", dpi: int = 300, output_format: str = "png"):
#     """One map with the storm's stations (agency-coded) and its best track."""
title = ""
dpi = 300
output_format = "png"
mapped = stations.dropna(subset=["lat", "lon"]).copy()
if mapped.empty:
    raise ValueError("No valid station coordinates")
else:
    print("ok")

extent = padded_extent(mapped, track)

projection = ccrs.PlateCarree()
fig = plt.figure(figsize=(10.5, 6.8), constrained_layout=False)
ax = fig.add_axes([0.05, 0.085, 0.70, 0.80], projection=projection)
ax.set_extent(extent, crs=projection)
ax.set_aspect("auto")
add_base_features(ax)

if track:
    ax.plot([p[1] for p in track], [p[0] for p in track], transform=projection,
            color="black", linewidth=2.2, zorder=3, alpha=0.9)

ax.gridlines(draw_labels=False, linewidth=0.28, color="#9aa5ad",
    alpha=0.42, linestyle=(0, (2, 3)))

for spine in ax.spines.values():
    spine.set_linewidth(0.55)
    spine.set_edgecolor("#4b5563")

plotted = [a for a in AGENCY_ORDER if a in set(mapped["agency"])]
plotted += sorted(set(mapped["agency"]) - set(plotted))
for agency in plotted:
    group = mapped[mapped["agency"] == agency]
    color = AGENCY_COLORS.get(str(agency), "#8a6d3b")
    obs = group[group["has_observed"]]
    fc = group[~group["has_observed"]]
    if not fc.empty:
        ax.scatter(fc["lon"], fc["lat"], transform=projection, s=60, marker="o",
                    facecolor=color, edgecolor="#1a1a1a", linewidth=0.3, zorder=5,
                    path_effects=[pe.withStroke(linewidth=1.0, foreground="#2f3b43", alpha=0.25)])
    if not obs.empty:
        ax.scatter(obs["lon"], obs["lat"], transform=projection, s=90, marker="^",
                    facecolor=color, edgecolor="#1a1a1a", linewidth=0.35, zorder=6,
                    path_effects=[pe.withStroke(linewidth=1.2, foreground="#2f3b43", alpha=0.3)])

legend_style = {"frameon": True, "framealpha": 0.95, "facecolor": "white",
                "edgecolor": "#d0d5da", "borderpad": 0.45, "labelspacing": 0.42,
                "handletextpad": 0.55, "fontsize": 8.6, "title_fontsize": 9.0}
agency_handles = [
    mlines.Line2D([], [], marker="^", linestyle="None", markersize=10,
                    markerfacecolor=AGENCY_COLORS.get(str(a), "#E53935"),
                    markeredgecolor="#1a1a1a", markeredgewidth=0.3, label=str(a))
    for a in plotted
]
data_handles = [
    mlines.Line2D([], [], marker="^", linestyle="None", markersize=9.5,
                    color="#30343b", label="Observed + forecast"),
    mlines.Line2D([], [], marker="o", linestyle="None", markersize=9,
                    color="#30343b", label="Forecast only"),
    mlines.Line2D([], [], color="black", linewidth=2.2, label="NOAA best track"),
]
agency_legend = ax.legend(handles=agency_handles, title="Agency", loc="center left",
                            bbox_to_anchor=(1.015, 0.62), **legend_style)
ax.add_artist(agency_legend)
ax.legend(handles=data_handles, title="Data", loc="center left",
            bbox_to_anchor=(1.015, 0.28), **legend_style)

fig.text(0.40, 0.945, title or "Stations + Best Track", ha="center", va="center",
        fontsize=18, fontweight="bold", color="#111827")
fig.text(0.40, 0.908,
        f"Stations: {len(mapped)} \u00b7 Track advisories: {len(track)}",
        ha="center", va="center", fontsize=11.5, color="#4b5563")

output_dir.mkdir(parents=True, exist_ok=True)
stem = clean_path_part(title) if title else clean_path_part(package_dir.name)
output_path = output_dir / f"{stem}_stations_track.{output_format}"
fig.savefig(output_path, dpi=dpi, facecolor="white")
plt.show()
plt.close(fig)
print("Saved:", output_path)
return output_path

ok


NameError: name 'title' is not defined

Error in callback <function _draw_all_if_interactive at 0x000001FEB58E4A40> (for post_execute), with arguments args (),kwargs {}:


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)>

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)>

<Figure size 1050x680 with 1 Axes>

## 7. Render and display

The map is exported and shown inline (displayed from the saved PNG, since a `cartopy` GeoAxes
does not render reliably through the inline `plt.show()` path).

In [ ]:
from IPython.display import Image, display

output_path = plot_stations_track(
    package_dir=PACKAGE_DIR,
    dat_path=DAT_PATH,
    output_dir=OUTPUT_DIR,
    title=TITLE,
    dpi=DPI,
    output_format=OUTPUT_FORMAT,
)
display(Image(filename=str(output_path)))

## 8. Done

The map is written to `OUTPUT_DIR`. To plot another case, change `PACKAGE_DIR`, `DAT_PATH`, and
`TITLE` in the **Input** cell and re-run from there.
